##  Image Classification for Astronomy

Astronomy as a field has embraced the wealth of data available.  Star spectra, deep-sky surveys, radio-frequency observations, and more have been collected for decades.  In fact, the [Large Synoptic Survey Telescope](https://www.lsst.org/) is expected to generate up to 30 terabytes of data nightly, or about 1.3 petabytes per year.  (The LSST's data management will be supported by [NCSA](http://www.ncsa.illinois.edu/).)

![](https://upload.wikimedia.org/wikipedia/commons/thumb/f/fb/Close_up_of_Telescope_in_the_Dome.jpg/640px-Close_up_of_Telescope_in_the_Dome.jpg)

Since the incoming rate of astronomical data has outstripped the ability of humans to review everything, machine learning techniques are employed to understand the data automatically.

One common task is to identify an object.  For instance, a bright spot may be a nearby star, a distant galaxy, or a remote quasar.  These have spectral differences which makes it possible to distinguish them.  We will use a _supervised learning_ algorithm to classify objects.  A supervised learning method uses a training data set to tell the program how to distinguish data; the program is then applied to new data.

In [ ]:
import numpy as np
from numpy import random as npr
import matplotlib.pyplot as plt
import matplotlib.cm as cm
%matplotlib inline

In [ ]:
train_data = np.load('./data/sdssdr6_colors_class_train.npy')
test_data = np.load('./data/sdssdr6_colors_class.200000.npy')

Let's examine one of the training data entries.

In [ ]:
train_data[ 0 ]
train_data.shape
train_data

What the five numbers represent are spectral intensities at different points in the electromagnetic spectrum, and the redshift.  The Sloan Digital Sky Survey, which provided these data, stores not full spectra, which are difficult to obtain, but composite single numbers giving one the gist of a particular celestial object without detailed study.  The bands are denoted by the letters `u` (ultraviolet), `g` (green), `r` (red), `i` (infrared), and `z` (far infrared).

![](./img/spectrum.png)

We have superimposed the spectrum of the star Sirius obtained from the [CALSPEC Calibration Database](http://www.stsci.edu/hst/observatory/crds/calspec.html) on the filter bands above.

Actually, if you examine the `dtype`, you can see that the array actually stores joint values or differences such as `u-g`.  This diminishes error due to imprecision in measurement.  The last number is the redshift.

Given these five numbers, we can train a classification tool to separate objects into two bins—stars and quasars—on the basis of the redshift.  (Quasars are strongly redshifted due to their distance.)

We have two data sets, one used for training and one for testing.  We will only use the first four numbers to build our model, and will use the redshift to classify objects.

The first classification tool we will use is the [_support vector machine_](http://scikit-learn.org/stable/modules/svm.html).  The SVM approach is good for high-dimensional spaces, even when there are more features than samples.  (That's not the case here, but could be if the data set has many possible features.)

In [ ]:
train_data = np.load('./data/sdssdr6_colors_class_train.npy')
test_data = np.load('./data/sdssdr6_colors_class.200000.npy')

The data are in the wrong shape to use with _scikit-learn_.  (This is a common problem.)

The data are also highly dimensional, making it difficult to visualize different clusters or zones.

In [ ]:
train_data_block = np.vstack( ( train_data[ 'u-g' ],train_data[ 'g-r' ],train_data[ 'r-i' ],train_data[ 'i-z' ] ) ).T

In [ ]:
train_data[ 'redshift' ][ :1000 ] > 1.0

In [ ]:
from sklearn import svm

clf = svm.SVC( kernel='rbf' )
clf.fit( train_data_block[ :20000,: ],train_data[ 'redshift' ][ :20000 ] > 1.0 )

In [ ]:
test_data_block = np.vstack( ( test_data[ 'u-g' ],test_data[ 'g-r' ],test_data[ 'r-i' ],test_data[ 'i-z' ] ) ).T

In [ ]:
train_data[ 'redshift' ][ :20000 ] > 1.0

In [ ]:
results_data_block = clf.predict( test_data_block )
sum( np.isclose( results_data_block,( test_data[ 'label' ] == 1 ) ) ) / len( results_data_block )

Various kernels can be used.  By assessing the similarity of the trained data to the actual classification, we can get a good idea which methods are appropriate.

TODO

In [ ]:
clf = svm.SVC( kernel='linear' )
clf.fit( train_data_block[ :20000,: ],train_data[ 'redshift' ][ :20000 ] > 1.0 )
results_data_block = clf.predict( test_data_block )
sum( np.isclose( results_data_block,( test_data[ 'label' ] == 1 ) ) ) / len( results_data_block )

In [ ]:
clf = svm.SVC( kernel='poly',degree=3 )
clf.fit( train_data_block[ :20000,: ],train_data[ 'redshift' ][ :20000 ] > 1.0 )
results_data_block = clf.predict( test_data_block )
sum( np.isclose( results_data_block,( test_data[ 'label' ] == 1 ) ) ) / len( results_data_block )

In [ ]:
from sklearn import naive_bayes

clf = naive_bayes.GaussianNB()
clf.fit( train_data_block[ :,: ],train_data[ 'redshift' ] > 0 )
results_data_block = clf.predict( test_data_block )
sum( np.isclose( results_data_block,( test_data[ 'label' ] == 1 ) ) ) / len( results_data_block )

-   Reimplement the training algorithm using different subsets of the training data.  Do these differ in their effectiveness?

-   TODO visualization something something mumble

This lesson drew from material developed by [Olivier Grisel](http://ogrisel.github.io/scikit-learn.org/sklearn-tutorial/tutorial/astronomy/classification.html) for _scikit-learn_.

## Appendix:  Plotting Filter Bands & Spectral Data

In [ ]:
# Produce plot of filter bands with Sirius spectral data.
# Taken from TODO
import os
import requests

import numpy as np
import pylab as pl
from matplotlib.patches import Arrow

from sklearn.datasets import get_data_home

import astropy.io.fits
data = astropy.io.fits.open( './data/sirius_stis_002.fits',lazy_load_hdus=False )
sirius_data = data[ 1 ]
sirius_data.data  # force it to load
Xref = sirius_data.columns[ 'WAVELENGTH' ].array
Yref = sirius_data.columns[ 'FLUX' ].array

#----------------------------------------------------------------------
# Plot filters in color with a single spectrum
fig,ax1 = pl.subplots()
ax1.semilogx( Xref,Yref,'-k',lw=1,color='gray' )
ax1.grid( False )
ax1.set_ylabel( 'normalized flux' )

ax2 = ax1.twinx()

import requests
colors = ( 'purple','green','red','maroon','black' )
for filter,color in zip( 'ugriz',colors ):
    url = f'http://www.sdss.org/dr7/instruments/imager/filters/{filter}.dat'
    data = requests.get( url ).text
    data_array = np.fromstring( '\n'.join( data.split( '\n' )[ 6: ] ).replace( '   ','\t' ),sep='\t' )
    array = np.reshape( data_array,( len( data_array )//5,5 ) )
    ax2.fill( array[ :,0 ],array[ :,1 ],ec=color,fc=color,alpha=0.4 )

kwargs = dict( fontsize=20,ha='center',va='center',alpha=0.5 )
pl.text( 3500,0.02,'u',color='purple',**kwargs )
pl.text( 4600,0.02,'g',color='green',**kwargs )
pl.text( 6100,0.02,'r',color='red',**kwargs )
pl.text( 7500,0.02,'i',color='maroon',**kwargs )
pl.text( 8800,0.02,'z',color='black',**kwargs )

ax2.set_ylabel( 'filter transmission' )
ax2.grid( False )

pl.title( 'SDSS Filters and Reference Spectrum' )
ax2.set_xlabel( 'Wavelength (Å)' )
pl.xlim( 3000,11000 )
pl.show()